In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

Цель A/B тестирования: определить стоит ли понизить сложность прохождения 2 уровня в главе 1.

Ниже создаем две таблицы:

*   для группы А оставляем уровень сложности
*   для группы Б понижаем уровень сложности

In [ ]:
oplogs = pd.read_excel('project.xlsx')
oplogs

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,param_2,extra_1,extra_2,level
0,android,1000001,a1000001ES1,2023-02-01 08:00:02,ES,1.01,app_install_start,0.0,NaN,organic,NaN,0
1,android,1000001,a1000001ES1,2023-02-01 08:00:02,ES,1.01,performance,27.0,NaN,samsung,galaxy a52,0
2,android,1000001,a1000001ES1,2023-02-01 08:01:16,ES,1.01,app_install_finish,74.0,NaN,NaN,NaN,0
3,android,1000001,a1000001ES1,2023-02-01 08:01:42,ES,1.01,loading_finish,26.0,NaN,NaN,NaN,0
4,android,1000001,a1000001ES1,2023-02-01 08:01:42,ES,1.01,tutorial_step_start,1.0,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
30984,iOS,1002499,i1002499GE1,2023-02-15 17:34:55,GE,1.02,tutorial_step_start,5.0,NaN,NaN,NaN,0
30985,iOS,1002499,i1002499GE1,2023-02-15 17:35:07,GE,1.02,tutorial_step_finish,5.0,NaN,NaN,NaN,0
30986,iOS,1002499,i1002499GE1,2023-02-15 17:35:07,GE,1.02,tutorial_step_start,6.0,NaN,NaN,NaN,0
30987,iOS,1002499,i1002499GE1,2023-02-15 17:35:11,GE,1.02,tutorial_step_finish,6.0,NaN,NaN,NaN,0


In [ ]:
# создание границ дат через переменную, в пределах которых проходил тест
start_date = pd.to_datetime('2023-02-03').date()
end_date = pd.to_datetime('2023-02-10').date()

In [ ]:
# дублирование столбца user_time, обрезая время
oplogs['user_date'] = oplogs['user_time'].dt.date

In [ ]:
# сокращение таблицы с учетом тестового периода
test_period_oplogs = oplogs[(oplogs['user_date'] >= start_date) & (oplogs['user_date'] <= end_date)]
test_period_oplogs

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,param_2,extra_1,extra_2,level,user_date
311,android,1000022,a1000022US3,2023-02-03 10:11:28,US,1.01,loading_finish,27.0,NaN,NaN,NaN,11,2023-02-03
312,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,chest_open,0.0,NaN,NaN,NaN,11,2023-02-03
313,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,res_movement_inc,500.0,3550.0,gold,chest,11,2023-02-03
314,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,res_movement_inc,10.0,15.0,token,chest,11,2023-02-03
315,android,1000022,a1000022US3,2023-02-03 10:12:04,US,1.01,res_movement_outc,3.0,12.0,token,stage_start,11,2023-02-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21338,android,1001679,a1001679KZ1,2023-02-10 23:34:57,KZ,1.02,res_movement_inc,100.0,950.0,gold,stage_win,5,2023-02-10
21339,android,1001679,a1001679KZ1,2023-02-10 23:35:09,KZ,1.02,res_movement_outc,1.0,5.0,token,stage_start,5,2023-02-10
21340,android,1001679,a1001679KZ1,2023-02-10 23:35:09,KZ,1.02,stage_start,5.0,1.0,bronze_loc,NaN,5,2023-02-10
21341,android,1001679,a1001679KZ1,2023-02-10 23:36:38,KZ,1.02,stage_lesion,5.0,1.0,bronze_loc,NaN,5,2023-02-10


In [ ]:
# проверка типа данных user_id
print(test_period_oplogs['user_id'].dtype)

int64


In [ ]:
# конвертирую user_id в string для применения регулярных выражений
test_period_oplogs['user_id'] = test_period_oplogs['user_id'].astype(str)
print(test_period_oplogs['user_id'].dtype)

object


<ipython-input-126-350a0b8efa11>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_period_oplogs['user_id'] = test_period_oplogs['user_id'].astype(str)


In [ ]:
# создаю тестовую группу А (user_id – заканчиваются на 0,1,2,3,4)
group_a = test_period_oplogs[test_period_oplogs['user_id'].str.endswith(('0', '1', '2', '3', '4'))]
group_a

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,param_2,extra_1,extra_2,level,user_date
311,android,1000022,a1000022US3,2023-02-03 10:11:28,US,1.01,loading_finish,27.0,NaN,NaN,NaN,11,2023-02-03
312,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,chest_open,0.0,NaN,NaN,NaN,11,2023-02-03
313,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,res_movement_inc,500.0,3550.0,gold,chest,11,2023-02-03
314,android,1000022,a1000022US3,2023-02-03 10:11:42,US,1.01,res_movement_inc,10.0,15.0,token,chest,11,2023-02-03
315,android,1000022,a1000022US3,2023-02-03 10:12:04,US,1.01,res_movement_outc,3.0,12.0,token,stage_start,11,2023-02-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21236,android,1001674,a1001674RU2,2023-02-10 23:41:18,RU,1.02,stage_start,5.0,1.0,bronze_loc,NaN,5,2023-02-10
21237,android,1001674,a1001674RU2,2023-02-10 23:42:17,RU,1.02,stage_win,5.0,1.0,bronze_loc,bronze_cup,5,2023-02-10
21238,android,1001674,a1001674RU2,2023-02-10 23:42:17,RU,1.02,lvl_up,6.0,NaN,NaN,NaN,6,2023-02-10
21239,android,1001674,a1001674RU2,2023-02-10 23:42:17,RU,1.02,res_movement_inc,1.0,1.0,bronze_cup,stage_win,6,2023-02-10


In [ ]:
# создаю тестовую группу B: user_id – заканчиваются на 5,6,7,8,9
group_b = test_period_oplogs[test_period_oplogs['user_id'].str.endswith(('5', '6', '7', '8', '9'))]
group_b

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,param_2,extra_1,extra_2,level,user_date
1070,iOS,1000076,i1000076RU3,2023-02-03 17:11:30,RU,1.01,loading_finish,21.0,NaN,NaN,NaN,11,2023-02-03
1071,iOS,1000076,i1000076RU3,2023-02-03 17:11:40,RU,1.01,chest_open,0.0,NaN,NaN,NaN,11,2023-02-03
1072,iOS,1000076,i1000076RU3,2023-02-03 17:11:40,RU,1.01,res_movement_inc,500.0,3550.0,gold,chest,11,2023-02-03
1073,iOS,1000076,i1000076RU3,2023-02-03 17:11:40,RU,1.01,res_movement_inc,10.0,15.0,token,chest,11,2023-02-03
1074,iOS,1000076,i1000076RU3,2023-02-03 17:12:02,RU,1.01,res_movement_outc,3.0,12.0,token,stage_start,11,2023-02-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21338,android,1001679,a1001679KZ1,2023-02-10 23:34:57,KZ,1.02,res_movement_inc,100.0,950.0,gold,stage_win,5,2023-02-10
21339,android,1001679,a1001679KZ1,2023-02-10 23:35:09,KZ,1.02,res_movement_outc,1.0,5.0,token,stage_start,5,2023-02-10
21340,android,1001679,a1001679KZ1,2023-02-10 23:35:09,KZ,1.02,stage_start,5.0,1.0,bronze_loc,NaN,5,2023-02-10
21341,android,1001679,a1001679KZ1,2023-02-10 23:36:38,KZ,1.02,stage_lesion,5.0,1.0,bronze_loc,NaN,5,2023-02-10


In [ ]:
# фильтрация группы А, где глава = 1, уровень = 2
group_a = group_a.query("event_name == 'stage_start' or event_name == 'stage_win' or event_name == 'stage_lesion'")\
.query("param_2 == 1")\
.query("level == 2")\
.rename(columns={'param_2': 'chapter_1'})
group_a

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,chapter_1,extra_1,extra_2,level,user_date
3760,android,1000303,a1000303US1,2023-02-03 00:50:50,US,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3761,android,1000303,a1000303US1,2023-02-03 00:51:41,US,1.01,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3866,android,1000311,a1000311US1,2023-02-03 02:02:06,US,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3867,android,1000311,a1000311US1,2023-02-03 02:02:52,US,1.01,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-03
4353,iOS,1000354,i1000354US1,2023-02-03 07:42:40,US,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20936,iOS,1001661,i1001661US1,2023-02-10 21:19:04,US,1.02,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21128,iOS,1001673,i1001673RU1,2023-02-10 22:48:17,RU,1.02,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21129,iOS,1001673,i1001673RU1,2023-02-10 22:49:01,RU,1.02,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21216,android,1001674,a1001674RU1,2023-02-10 22:57:10,RU,1.02,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-10


In [ ]:
# подсчет количества стартов уровня, успешного завершения, неуспешного завершения
group_a['event_name'].value_counts()

,count
event_name,
stage_start,101
stage_win,86
stage_lesion,15


In [ ]:
# фильтрация группы Б, где глава = 1, уровень = 2
group_b = group_b.query("event_name == 'stage_start' or event_name == 'stage_win' or event_name == 'stage_lesion'")\
.query("param_2 == 1")\
.query("level == 2")\
.rename(columns={'param_2': 'chapter_1'})
group_b

,platform_type,user_id,session_id,user_time,country,version,event_name,param_1,chapter_1,extra_1,extra_2,level,user_date
3712,android,1000299,a1000299IT1,2023-02-03 00:30:47,IT,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3713,android,1000299,a1000299IT1,2023-02-03 00:31:35,IT,1.01,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3942,iOS,1000317,i1000317RU1,2023-02-03 02:48:43,RU,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
3943,iOS,1000317,i1000317RU1,2023-02-03 02:49:36,RU,1.01,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-03
4036,android,1000327,a1000327IT1,2023-02-03 03:45:06,IT,1.01,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20983,android,1001665,a1001665PL1,2023-02-10 21:54:27,PL,1.02,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21274,android,1001675,a1001675US1,2023-02-10 23:02:43,US,1.02,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21275,android,1001675,a1001675US1,2023-02-10 23:03:31,US,1.02,stage_win,2.0,1.0,bronze_loc,NaN,2,2023-02-10
21325,android,1001679,a1001679KZ1,2023-02-10 23:32:19,KZ,1.02,stage_start,2.0,1.0,bronze_loc,NaN,2,2023-02-10


In [ ]:
# подсчет количества стартов уровня, успешного завершения, неуспешного завершения
group_b['event_name'].value_counts()

,count
event_name,
stage_start,107
stage_win,97
stage_lesion,10


In [ ]:
# присваиваем через переменную кол-во стартов и выигрышей для расчета win rate
win_group_a = len(group_a[group_a['event_name']=='stage_win'])
start_group_a = len(group_a[group_a['event_name']=='stage_start'])

win_group_b = len(group_b[group_b['event_name']=='stage_win'])
start_group_b = len(group_b[group_b['event_name']=='stage_start'])

print(f"Группа А: выигрыши = {win_group_a}, попытки = {start_group_a}")
print(f"Группа Б: выигрыши = {win_group_b}, попытки = {start_group_b}")

Группа А: выигрыши = 86, попытки = 101
Группа Б: выигрыши = 97, попытки = 107


In [ ]:
# расчет win rate
win_rate_a = round(win_group_a / start_group_a * 100, 2)
win_rate_b = round(win_group_b / start_group_b * 100, 2)

print(f"Группа А: win rate = {win_rate_a}")
print(f"Группа Б: win rate = {win_rate_b}")

Группа А: win rate = 85.15
Группа Б: win rate = 90.65


Для проведения исследования используем **z-test** – статистический тест, который используется для проверки гипотезы о среднем значении или пропорции в популяции, когда размер выборки достаточно велик (обычно более 30), и данные предполагают нормальное распределение. В основе Z-теста лежит сравнение среднего значения выборки с известным средним значением генеральной совокупности или других выборок.

**Нулевая гипотеза:** пропорции двух групп равны – следовательно уровень не понижаем

**Альтернативная гипотеза:** пропорции двух групп не равны – сложность 2-го уровня следует понизить

In [ ]:
#wins - массив из значений выигрышей для двух групп выигрышей, attempts - массив из значений попыток для двух групп
wins = np.array([win_group_a, win_group_b])
attempts = np.array([start_group_a, start_group_b])

In [ ]:
stat, pval = sm.stats.proportions_ztest(wins, attempts)
print(stat, pval)

-1.220395249028071 0.22231507752087898


In [ ]:
pval > 0.05

True

Так как pval > 0,05 – нулевая гипотеза не отвергается.

**Вывод:** 2 уровень в главе 1 не понижаем, так как это ни на что не влияет.